# GreenTest: C#

This is the **C#** GreenTest notebook. Every language in this repo follows the
same pattern: bootstrap the language, generate a small static site, serve
it locally, and verify it's actually being served correctly.

This one verifies [vanilla-compost](https://github.com/EcologyComputing/vanilla-compost) using C#.
The C# work lives in `greentest.cs`, a single-file C# app (`dotnet run
greentest.cs`, no `.csproj` needed). This notebook walks you through what it
does and runs it from a bash cell, the same way js's notebook shells out to
`node`.

**Start with python's GreenTest first.** `python/greentest.ipynb` covers the
one-time, per-machine setup every language relies on - git identity and
GitHub authentication - so it isn't repeated here. This notebook's
`bootstrap.sh` still installs its own python3 and Jupyter (in `.venv`
inside this folder), so it runs on its own either way.

Steps:
0. Confirm the .NET SDK is installed
1. Point this notebook at your vanilla-compost clone
2. Confirm it's actually cloned
3. Run the C# GreenTest (`greentest.cs` handles leaving a note, generating
   `posts.html`, serving it locally, verifying it, and cleaning up - all in
   one step, since it's a single self-contained script)
4. Peek at what got generated

If you haven't already, run `bootstrap.sh` in this same directory first -
it installs the .NET SDK and Jupyter, then opens this notebook.

## 0. Confirm the .NET SDK is installed

`greentest.cs` is a file-based app, which needs .NET SDK 10 or newer. This
bash cell checks that `dotnet` is on your PATH, actually runs (it crashes
without `libicu`, which `bootstrap.sh` installs), and is new enough.

In [ ]:
%%bash
if ! which dotnet >/dev/null 2>&1; then
    echo "dotnet was not found on PATH. Run ./bootstrap.sh in this directory,"
    echo "or open a new terminal (so ~/.bashrc puts ~/.dotnet on PATH) and relaunch Jupyter."
    exit 1
fi

if ! VERSION=$(dotnet --version 2>&1); then
    echo "dotnet is installed at $(which dotnet) but won't run:"
    echo "$VERSION"
    echo "Re-run ./bootstrap.sh - it installs libicu, which .NET needs to start."
    exit 1
fi

if [ "$(echo "$VERSION" | cut -d. -f1)" -lt 10 ]; then
    echo "Found .NET SDK $VERSION, but greentest.cs needs SDK 10 or newer. Re-run ./bootstrap.sh."
    exit 1
fi

echo "Found .NET SDK $VERSION at $(which dotnet)"

## 1. Point this notebook at your vanilla-compost clone

GreenTest assumes you cloned vanilla-compost as a sibling of this repo, so
the same folder that has `greenTest/` should also have `vanilla-compost/`.
This cell runs in **bash**, same as python's and js's notebooks. If your
clone of vanilla-compost lives somewhere else, change the path below.

In [ ]:
%%bash
export VANILLA_COMPOST="../../vanilla-compost"
echo "Testing vanilla-compost at: $VANILLA_COMPOST"

## 2. Confirm the repo is cloned

This next script is also in bash. It checks that the files `greentest.cs`
needs exist (`html_template.html` and the `posts/` directory - there's no
`generate_posts.cs` in vanilla-compost to check for, since C# implements
that logic directly in `greentest.cs`), and that the folder is a real
`git clone`.

In [ ]:
%%bash
VANILLA_COMPOST="../../vanilla-compost"
if [ -f "$VANILLA_COMPOST/README.md" ] && [ -f "$VANILLA_COMPOST/src/html_template.html" ] && [ -d "$VANILLA_COMPOST/src/posts" ]; then
    echo "Repo layout looks right (found README.md, src/html_template.html, src/posts/)."
else
    echo "That path doesn't look like a vanilla-compost clone: $VANILLA_COMPOST"
    exit 1
fi

if git -C "$VANILLA_COMPOST" rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    echo "Confirmed: this is a real git clone, not just a folder of files."
    git -C "$VANILLA_COMPOST" remote get-url origin 2>/dev/null && echo "Its 'origin' remote points there." || echo "No 'origin' remote set."
else
    echo "Warning: no .git found there."
fi

## 3. Run the C# GreenTest

This is the actual test. `greentest.cs` does everything python's and js's
notebooks do across several cells, in one self-contained run: leaves a
timestamped note in `greenTest-Message.md` (copied into vanilla-compost's
`posts/`), generates `posts.html`, serves it locally, fetches it back to
verify it matches, then cleans up.

We run it with `--quick` here since a notebook's bash cell isn't an
interactive terminal - `--quick` skips the "enter notes for this run"
prompt in favor of a default note, and trims the narration `greentest.cs`
prints when run directly from the command line. For the full narrated
walkthrough with your own notes, run `./greentest.cs` directly in a
terminal instead.

In [ ]:
%%bash
set -e
export VANILLA_COMPOST="../../vanilla-compost"
./greentest.cs --quick

## 4. Peek at what got generated

`greentest.cs` already verified the server was serving the freshly
generated `posts.html` correctly (that's the "Green: ..." line above) and
shut the server down afterward. This cell independently confirms the
*file itself* was written correctly, the same way python's and js's
notebooks peek at their own generated output.

In [ ]:
import os

VANILLA_COMPOST = os.environ.get("VANILLA_COMPOST", "../../vanilla-compost")
posts_path = os.path.join(VANILLA_COMPOST, "src", "posts.html")

with open(posts_path, encoding="utf-8") as f:
    generated = f.read()

assert "hello-compost" in generated, "Expected the hello-compost post to be listed."

start = generated.find('<p class="lead">')
end = generated.find('</p>', start) + len('</p>')
print(generated[start:end] if start != -1 else generated)

If everything above ran without errors, C# is bootstrapped and working
end to end on this machine, verified against a small app instead of just
assumed to be working. See `greenTest-Message.md` for a running history of
these runs.

See `../ECOLOGY.md` for how this fits into the rest of the Ecology
Computing methodology.